# MobileViT-XS baseline

This notebook fine-tunes the public ImageNet-pretrained MobileViT-XS checkpoint on the deterministic development manifest. It performs a frozen-head warm-up followed by full-model fine-tuning and retains the checkpoint with the best validation F1.

> **Scope:** these are preliminary pipeline-validation results. The manifest uses a tile-level split and may contain parent-photograph leakage; do not report it as a Phase 3 research result.


## 1. Configuration

The dataset paths come from the data-audit notebook.


In [ ]:
from collections import Counter
from contextlib import nullcontext
import csv
import json
from pathlib import Path
import random

import numpy as np
from PIL import Image
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
import timm
from timm.data import create_transform, resolve_model_data_config
from timm.data.transforms import str_to_interp_mode
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
MODEL_NAME = "mobilevit_xs.cvnets_in1k"
BATCH_SIZE = 32
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 5
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4

AUDIT_DIR = Path("outputs/data_audit")
MANIFEST_PATH = AUDIT_DIR / "development_manifest.csv"
AUDIT_SUMMARY_PATH = AUDIT_DIR / "data_audit_summary.json"
OUTPUT_DIR = Path("outputs/mobilevit_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print({"device": str(DEVICE), "torch": torch.__version__, "timm": timm.__version__})


## 2. Load and verify the preliminary development split


In [ ]:
if not MANIFEST_PATH.exists() or not AUDIT_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "Data-audit outputs are missing. Run notebooks/01_data_audit.ipynb first."
    )

audit_summary = json.loads(AUDIT_SUMMARY_PATH.read_text(encoding="utf-8"))
dataset_root = Path(audit_summary["dataset_root"])
with MANIFEST_PATH.open(newline="", encoding="utf-8") as stream:
    rows = list(csv.DictReader(stream))

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root no longer exists: {dataset_root}. Rerun the data-audit notebook.")
if not rows or {row["split_basis"] for row in rows} != {"tile_development_only"}:
    raise ValueError("Expected the preliminary tile_development_only manifest.")

train_rows = [row for row in rows if row["split"] == "dev_train"]
validation_rows = [row for row in rows if row["split"] == "dev_validation"]
assert train_rows and validation_rows
assert not ({row["path"] for row in train_rows} & {row["path"] for row in validation_rows})

counts = Counter((row["split"], int(row["label"]), row["surface"]) for row in rows)
for key, value in sorted(counts.items()):
    print(key, value)
print(f"Train: {len(train_rows):,} | Validation: {len(validation_rows):,}")
print("WARNING: tile_development_only; possible parent-photo leakage.")


## 3. Resolve checkpoint preprocessing and build loaders

Resize, crop, interpolation, normalization, and channel handling come from the checkpoint configuration. Training adds only the planned crack-safe augmentation.


In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=1)
data_config = resolve_model_data_config(model)

base_train_transform = create_transform(
    **data_config,
    is_training=True,
    scale=(0.85, 1.0),
    hflip=0.5,
    vflip=0.5,
    color_jitter=0.2,
)
train_transform = transforms.Compose([
    transforms.RandomRotation(
        15, interpolation=str_to_interp_mode(data_config["interpolation"])
    ),
    base_train_transform,
])
validation_transform = create_transform(**data_config, is_training=False)

class CrackDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        row = self.items[index]
        with Image.open(dataset_root / row["path"]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.tensor(float(row["label"])), row["path"]

train_loader = DataLoader(
    CrackDataset(train_rows, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    CrackDataset(validation_rows, validation_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=DEVICE.type == "cuda",
)

sample_images, sample_labels, _ = next(iter(train_loader))
print(json.dumps(data_config, indent=2, default=list))
print("Batch:", tuple(sample_images.shape), tuple(sample_labels.shape))
assert tuple(sample_images.shape[1:]) == tuple(data_config["input_size"])
with torch.inference_mode():
    smoke_logits = model(sample_images[:2])
assert smoke_logits.shape == (2, 1)
assert torch.isfinite(smoke_logits).all()
print("Forward check:", tuple(smoke_logits.shape), "parameters:", sum(p.numel() for p in model.parameters()))


## 4. Loss, metrics, and one training loop

`pos_weight` is calculated from the training partition only. No balanced sampler is combined with the weighted loss.


In [ ]:
train_positive = sum(int(row["label"]) == 1 for row in train_rows)
train_negative = len(train_rows) - train_positive
pos_weight = torch.tensor([train_negative / train_positive], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
model = model.to(DEVICE)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

def amp_context():
    return torch.amp.autocast("cuda") if DEVICE.type == "cuda" else nullcontext()

def binary_metrics(labels, probabilities):
    predictions = (probabilities >= 0.5).astype(np.int64)
    return {
        "f1": f1_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "precision": precision_score(labels, predictions, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "pr_auc": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "confusion_matrix": confusion_matrix(labels, predictions).tolist(),
    }

def run_epoch(loader, optimizer=None, head_only=False):
    training = optimizer is not None
    if training and head_only:
        model.eval()
        model.get_classifier().train()
    elif training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_labels, all_probabilities = [], []
    for images, labels, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training), amp_context():
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_probabilities.extend(torch.sigmoid(logits).detach().cpu().numpy())

    labels_array = np.asarray(all_labels, dtype=np.int64)
    probabilities_array = np.asarray(all_probabilities)
    return total_loss / len(loader.dataset), binary_metrics(labels_array, probabilities_array)

print({"train_positive": train_positive, "train_negative": train_negative, "pos_weight": pos_weight.item()})


## 5. Phase 1 — train the new classifier head for three epochs


In [ ]:
CHECKPOINT_PATH = OUTPUT_DIR / "best_mobilevit_xs.pt"
HISTORY_PATH = OUTPUT_DIR / "history.json"
history = []
best_f1 = -1.0

def train_phase(name, epochs, learning_rate, head_only):
    global best_f1
    for parameter in model.parameters():
        parameter.requires_grad = not head_only
    if head_only:
        for parameter in model.get_classifier().parameters():
            parameter.requires_grad = True

    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=learning_rate, weight_decay=WEIGHT_DECAY,
    )
    for epoch in range(1, epochs + 1):
        train_loss, train_metrics = run_epoch(train_loader, optimizer, head_only=head_only)
        validation_loss, validation_metrics = run_epoch(validation_loader)
        record = {
            "phase": name, "epoch": epoch,
            "train_loss": train_loss, "validation_loss": validation_loss,
            "train_metrics": train_metrics,
            "validation_metrics": validation_metrics,
        }
        history.append(record)
        print(
            f"{name} {epoch}/{epochs} | train loss {train_loss:.4f} | "
            f"val loss {validation_loss:.4f} | val F1 {validation_metrics['f1']:.4f} | "
            f"val recall {validation_metrics['recall']:.4f}"
        )
        if validation_metrics["f1"] > best_f1:
            best_f1 = validation_metrics["f1"]
            torch.save({
                "model_name": MODEL_NAME,
                "model_state_dict": model.state_dict(),
                "data_config": data_config,
                "validation_metrics": validation_metrics,
                "phase": name, "epoch": epoch, "seed": SEED,
                "split_basis": "tile_development_only",
            }, CHECKPOINT_PATH)
        HISTORY_PATH.write_text(json.dumps(history, indent=2) + "\n", encoding="utf-8")

train_phase("head", HEAD_EPOCHS, HEAD_LR, head_only=True)
print(f"Best validation F1 after head training: {best_f1:.4f}")


## 6. Phase 2 — unfreeze and fine-tune the complete model


In [ ]:
train_phase("full", FINETUNE_EPOCHS, FINETUNE_LR, head_only=False)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print("Saved best checkpoint:", CHECKPOINT_PATH.resolve())
print(json.dumps(checkpoint["validation_metrics"], indent=2))
print("WARNING: tile_development_only; possible parent-photo leakage; not a Phase 3 result.")


## Outputs and limitations

The baseline run is complete when all eight epochs finish, `best_mobilevit_xs.pt` and `history.json` exist, validation loss and F1 are printed, and the output remains labelled `tile_development_only`. The evaluation notebook loads the saved checkpoint and produces figures and error analysis.
